In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.video import VIDEO_EXTENSIONS, get_video_metadata  # noqa: E402

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'RWF-2000'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir exists: {DATA_DIR.exists()}')

sns.set_theme(style='whitegrid')

In [ ]:
records: list[dict] = []

for split_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        videos = sorted(
            p for p in class_dir.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS
        )
        for video_path in videos:
            meta = get_video_metadata(video_path)
            if meta is None:
                continue

            stem = video_path.stem
            source_group = stem.rsplit('_', 1)[0] if '_' in stem else stem

            records.append(
                {
                    'split': split_dir.name,
                    'label': class_dir.name,
                    'filename': video_path.name,
                    'path': str(video_path),
                    'source_group': source_group,
                    'file_size_mb': video_path.stat().st_size / (1024 * 1024),
                    'fps': meta.fps,
                    'frame_count': meta.frame_count,
                    'duration_sec': meta.duration_sec,
                    'width': meta.width,
                    'height': meta.height,
                    'resolution': meta.resolution,
                    'aspect_ratio': meta.aspect_ratio,
                }
            )

df = pd.DataFrame(records)
print(f'Valid clips: {len(df)}')

In [ ]:
pd.crosstab(df['split'], df['label'], margins=True)

In [ ]:
df[['fps', 'frame_count', 'duration_sec', 'width', 'height', 'file_size_mb']].describe().round(2)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='split', hue='label')
ax.set_title('Class Balance by Split')
ax.set_xlabel('Split')
ax.set_ylabel('Clip Count')
ax.legend(title='Class')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rwf2000_class_balance.png')
plt.show()

In [ ]:
top_n = 10
counts = df['resolution'].value_counts()
top_values = counts.head(top_n).index
n_other = len(counts) - len(top_values)
other_label = f'Other ({n_other} unique)'
order = [*top_values, other_label]

grouped = df['resolution'].where(df['resolution'].isin(top_values), other_label)

plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df.assign(res_grouped=grouped),
    y='res_grouped',
    hue='label',
    order=order,
)

ax.set_title(f'Top {top_n} Video Resolutions')
ax.set_xlabel('Clip Count')
ax.set_ylabel('')
ax.legend(title='Class')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rwf2000_resolution_distribution.png', dpi=150)
plt.show()

In [ ]:
top_n = 10
ratios = df['aspect_ratio'].map('{:.2f}'.format)
counts = ratios.value_counts()
top_values = counts.head(top_n).index
n_other = len(counts) - len(top_values)
other_label = f'Other ({n_other} unique)'
order = [*top_values, other_label]

grouped = ratios.where(ratios.isin(top_values), other_label)

plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df.assign(ratio_grouped=grouped),
    y='ratio_grouped',
    hue='label',
    order=order
)

ax.set_title(f'Top {top_n} Aspect Ratios')
ax.set_xlabel('Clip Count')
ax.set_ylabel('')
ax.legend(title='Class')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rwf2000_aspect_ratio_distribution.png')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.histplot(
    data=df,
    x='file_size_mb',
    hue='label',
    bins=50,
    multiple='stack'
)

ax.set_title('File Size Distribution')
ax.set_xlabel('File Size [MB]')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rwf2000_file_size_distribution.png')
plt.show()

In [ ]:
group_counts = df['source_group'].value_counts()
multi_clip_groups = group_counts[group_counts > 1]
mixed_label_groups = df.groupby('source_group')['label'].nunique()
mixed_label_groups = mixed_label_groups[mixed_label_groups > 1]

print(f'Total valid clips: {len(df)}')
print(f'Unique source_group prefixes: {len(group_counts)}')
print(f'Groups with >1 clip: {len(multi_clip_groups)} ({multi_clip_groups.sum()} clips)')
print(f'Max clips per single group: {group_counts.max()}')
print(f'Groups containing both Fight and NonFight: {len(mixed_label_groups)}')

train_groups = set(df.loc[df['split'] == 'train', 'source_group'])
val_groups = set(df.loc[df['split'] == 'val', 'source_group'])
overlap = sorted(train_groups & val_groups)
print(f'Overlapping groups between original train and val: {len(overlap)}')

In [ ]:
def read_middle_frame(video_path: str):
    cap = cv2.VideoCapture(video_path)
    try:
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(total_frames // 2, 0))
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None
    finally:
        cap.release()


def plot_frame_rows(
    row_groups: list[tuple[str, pd.DataFrame]],
    save_name: str,
) -> None:
    nrows = len(row_groups)
    ncols = max(len(group_df) for _, group_df in row_groups)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.3 * nrows), squeeze=False)

    for r_idx, (row_title, group_df) in enumerate(row_groups):
        for c_idx in range(ncols):
            ax = axes[r_idx, c_idx]
            ax.axis('off')
            if c_idx >= len(group_df):
                continue

            row = group_df.iloc[c_idx]
            frame = read_middle_frame(row['path'])
            if frame is not None:
                ax.imshow(frame)
            ax.set_title(
                f"{row['label']} | {row['resolution']} | {row['file_size_mb']:.2f} MB",
                fontsize=9,
            )
            if c_idx == 0:
                ax.text(
                    -0.08,
                    0.5,
                    row_title,
                    transform=ax.transAxes,
                    rotation=90,
                    va='center',
                    ha='right',
                    fontsize=10,
                    fontweight='bold',
                )

    plt.tight_layout()
    out_path = FIGURES_DIR / save_name
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved frame grid to: {out_path}')


random_groups = [
    ('Fight', df[df['label'] == 'Fight'].sample(n=4, random_state=42)),
    ('NonFight', df[df['label'] == 'NonFight'].sample(n=4, random_state=42)),
]
plot_frame_rows(random_groups, 'rwf2000_sample_frames.png')

In [ ]:
def load_storyboard_frames(video_path: str, n_frames: int):
    cap = cv2.VideoCapture(video_path)
    frames = []
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()

    if not frames:
        return []
    indices = [i * (len(frames) - 1) // (n_frames - 1) for i in range(n_frames)]
    return [(idx, frames[idx]) for idx in indices]


n_frames = 12
n_cols = (n_frames + 1) // 2

fig = plt.figure(figsize=(15, 9), layout='constrained')
subfigs = fig.subfigures(2, 1, hspace=0.08)

for subfig, label in zip(subfigs, ['Fight', 'NonFight']):
    sample = df[df['label'] == label].sample(n=1).iloc[0]
    sampled = load_storyboard_frames(sample['path'], n_frames=n_frames)

    subfig.suptitle(
        f"{label} | {sample['filename']}",
        fontsize=12,
        fontweight='bold',
    )

    axes = subfig.subplots(2, n_cols)
    for idx, ax in enumerate(axes.flat):
        ax.axis('off')
        if idx < len(sampled):
            frame_idx, frame = sampled[idx]
            ax.imshow(frame)
            ax.set_title(f'{frame_idx / sample["fps"]:.2f}s', fontsize=9)

plt.show()